# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

##### **Lane:** Lane 2 — Refresh / Content Opportunity Scoring
##### 
##### **Goal:** Build and evaluate a model that beats the Week-4 baseline on the same data and same metric.
##### 
##### **Date:** 2026-08-07

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# <font size="1">
# ### Method: Random Forest Classifier
# 
# **Why Random Forest?**
# 
# | Reason | Explanation |
# |--------|-------------|
# | **Handles non-linearity** | Page decline doesn't follow simple linear patterns. Stale + high volume together matters more than either alone. |
# | **Feature interactions** | Random Forest automatically captures interactions (e.g., age × impressions × position). |
# | **Feature importance** | Provides interpretable feature importance scores — I can explain what the model learned. |
# | **Handles mixed data types** | Works with both numeric (impressions, position) and categorical (tiers) features. |
# | **Robust to outliers** | Tree-based models are less sensitive to outliers than logistic regression. |
# | **Proven on this data** | The starter pipeline already showed Random Forest achieving 0.740 Precision@50. |
# 
# **Alternatives Considered:**
# 
# | Model | Why Not Chosen |
# |-------|----------------|
# | **Logistic Regression** | Too simple for this problem. Can't capture non-linear patterns or interactions well. |
# | **Decision Tree** | Prone to overfitting. A single tree is less stable than an ensemble. |
# | **XGBoost** | More powerful but harder to interpret. Random Forest is sufficient for now; XGBoost can be tried later. |
# 
# **What I'm Predicting:**
# 
# > **Target:** `is_declining` (binary: 1 if `trend_direction == "down"`, else 0)
# 
# **Why this target:**
# - The starter dataset has a precomputed `trend_direction` column
# - A page is "declining" if its trend direction is "down"
# - This is a proxy for future decline — I'll upgrade to a future-looking target in Week 6
# 
# **Success Metric:**
# 
# > **Primary metric:** **Precision@50** — matches the real decision (reviewer checks top 50 pages)
# 
# **Why Precision@50:**
# - A content reviewer typically checks ~50 pages per week
# - Precision@50 measures: of those 50, how many are actually declining?
# - This matches the business problem exactly
# 
# **Target to beat:** Baseline Precision@50 = 0.240 (12/50 correct)
# 
# </font>

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load starter dataset
try:
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
except FileNotFoundError:
    try:
        df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
    except FileNotFoundError:
        df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

print(f"Loaded {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

Loaded 30,000 rows
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# <font size="1">
# ### Validation Design: Client-Holdout
# 
# **Why client-holdout?**
# 
# | Reason | Explanation |
# |--------|-------------|
# | **Tests generalization** | A model that memorizes client-specific patterns would fail on new clients. Client-holdout tests whether the model actually generalizes. |
# | **Matches reality** | In production, the model will be applied to new clients it hasn't seen before. |
# | **Prevents leakage** | Pages from the same client share underlying patterns (industry, content strategy, audience). Keeping them together prevents leakage. |
# 
# **Split Strategy:**
# - 70% of clients in train set
# - 30% of clients in test set
# - All pages from a client stay together (no mixing)
# 
# </font>


In [5]:
# Check client distribution
clients = df['client_id'].unique()
print(f"Total clients: {len(clients)}")
print(f"Client IDs: {clients}")

Total clients: 32
Client IDs: ['client_f369cb89fc' 'client_4e07408562' 'client_7f2253d7e2'
 'client_19581e27de' 'client_3fdba35f04' 'client_8722616204'
 'client_6208ef0f77' 'client_d4735e3a26' 'client_8527a891e2'
 'client_9f14025af0' 'client_349c41201b' 'client_e629fa6598'
 'client_4ec9599fc2' 'client_d59eced1de' 'client_2c624232cd'
 'client_bbb965ab0c' 'client_d029fa3a95' 'client_a88a7902cb'
 'client_98a3ab7c34' 'client_f74efabef1' 'client_624b60c58c'
 'client_e29c9c180c' 'client_25fc0e7096' 'client_b4944c6ff0'
 'client_9400f1b21c' 'client_434c9b5ae5' 'client_0b918943df'
 'client_1a6562590e' 'client_bdd2d3af3a' 'client_8b940be7fb'
 'client_02d20bbd7e' 'client_4fc82b26ae']


In [6]:
# Show client page counts
client_counts = df.groupby('client_id').size().sort_values(ascending=False)
print(f"\nPages per client (top 10):")
print(client_counts.head(10))


Pages per client (top 10):
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
dtype: int64


In [7]:
# Create client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)


In [8]:
# Get train/test indices by client
for train_idx, test_idx in gss.split(df, groups=df['client_id']):
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

In [9]:
print(f"\nTrain set: {len(train_df):,} pages, {train_df['client_id'].nunique()} clients")
print(f"Test set: {len(test_df):,} pages, {test_df['client_id'].nunique()} clients")
print(f"Train clients: {sorted(train_df['client_id'].unique())}")
print(f"Test clients: {sorted(test_df['client_id'].unique())}")


Train set: 19,166 pages, 22 clients
Test set: 10,834 pages, 10 clients
Train clients: ['client_0b918943df', 'client_19581e27de', 'client_1a6562590e', 'client_25fc0e7096', 'client_2c624232cd', 'client_349c41201b', 'client_3fdba35f04', 'client_4ec9599fc2', 'client_4fc82b26ae', 'client_624b60c58c', 'client_7f2253d7e2', 'client_8722616204', 'client_9400f1b21c', 'client_98a3ab7c34', 'client_9f14025af0', 'client_a88a7902cb', 'client_b4944c6ff0', 'client_bbb965ab0c', 'client_d4735e3a26', 'client_d59eced1de', 'client_e29c9c180c', 'client_f74efabef1']
Test clients: ['client_02d20bbd7e', 'client_434c9b5ae5', 'client_4e07408562', 'client_6208ef0f77', 'client_8527a891e2', 'client_8b940be7fb', 'client_bdd2d3af3a', 'client_d029fa3a95', 'client_e629fa6598', 'client_f369cb89fc']


In [10]:
# Check label distribution
print(f"\nLabel distribution in train set:")
print(train_df['trend_direction'].value_counts())
print(f"\nLabel distribution in test set:")
print(test_df['trend_direction'].value_counts())


Label distribution in train set:
trend_direction
down      10201
stable     3555
up         2742
new        1918
flat        750
Name: count, dtype: int64

Label distribution in test set:
trend_direction
down      6061
stable    2407
up        1646
flat       402
new        318
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

# <font size="1">
# ### Feature Engineering
# 
# </font>


In [11]:
# Select features for the model
feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
    'content_age_days', 'engagement_rate', 'scroll_rate',
    'position_tier', 'age_tier', 'impression_tier'
]

In [12]:
# Check which features exist
available_features = [col for col in feature_cols if col in df.columns]
print(f"Available features: {available_features}")

Available features: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'engagement_rate', 'scroll_rate', 'position_tier', 'age_tier', 'impression_tier']


In [13]:
# Create target: binary label from trend_direction
train_df['target'] = (train_df['trend_direction'] == 'down').astype(int)
test_df['target'] = (test_df['trend_direction'] == 'down').astype(int)


In [14]:
print(f"\nTarget distribution:")
print(f"Train: {train_df['target'].value_counts().to_dict()}")
print(f"Test: {test_df['target'].value_counts().to_dict()}")


Target distribution:
Train: {1: 10201, 0: 8965}
Test: {1: 6061, 0: 4773}


In [15]:
# Prepare features
X_train = train_df[available_features].copy()
X_test = test_df[available_features].copy()
y_train = train_df['target']
y_test = test_df['target']

In [16]:
# Handle missing values
for col in X_train.columns:
    if X_train[col].dtype in ['int64', 'float64']:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
    else:
        mode_val = X_train[col].mode()[0] if not X_train[col].mode().empty else 'unknown'
        X_train[col] = X_train[col].fillna(mode_val)
        X_test[col] = X_test[col].fillna(mode_val)


In [17]:
print(f"\nFeatures shape: {X_train.shape[1]} features")
print(f"Train: {X_train.shape[0]:,} rows")
print(f"Test: {X_test.shape[0]:,} rows")


Features shape: 10 features
Train: 19,166 rows
Test: 10,834 rows


# <font size="1">
# 
# ### Baseline (from Week 4)
# 
# </font>

In [18]:
print("BASELINE: Week-4 Rule")
print("=" * 60)

def calculate_baseline_score(row):
    """Calculate baseline refresh score from Week 4."""
    staleness_score = 1.0 if row['days_since_last_update'] >= 180 else 0.0
    volume_score = min(row['impressions_90d'] / 10000, 1.0)
    
    if row['avg_position'] <= 5:
        position_score = 1.0
    elif row['avg_position'] <= 10:
        position_score = 0.5
    else:
        position_score = 0.0
    
    score = 0.40 * staleness_score + 0.35 * volume_score + 0.25 * position_score
    return score


BASELINE: Week-4 Rule


In [19]:
# Apply baseline to test set
test_df['baseline_score'] = test_df.apply(calculate_baseline_score, axis=1)

In [20]:
# Calculate Precision@50
top_k = 50
test_sorted_baseline = test_df.sort_values('baseline_score', ascending=False)
top_baseline = test_sorted_baseline.head(top_k)
baseline_precision = top_baseline['target'].mean()


In [21]:
print(f"Baseline Precision@{top_k}: {baseline_precision:.3f} ({int(baseline_precision * top_k)}/{top_k} correct)")

Baseline Precision@50: 0.440 (22/50 correct)


# <font size="1">
# 
# ### Model 1: Logistic Regression
# 
# </font>

In [22]:
print("MODEL 1: Logistic Regression")
print("=" * 60)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)


MODEL 1: Logistic Regression


ValueError: could not convert string to float: 'page_3_5'

In [ ]:
y_test_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]


In [ ]:
test_df_lr = test_df.copy()
test_df_lr['pred_proba'] = y_test_pred_proba_lr
test_sorted_lr = test_df_lr.sort_values('pred_proba', ascending=False)
top_lr = test_sorted_lr.head(top_k)
lr_precision = top_lr['target'].mean()

In [ ]:
lr_accuracy = lr_model.score(X_test_scaled, y_test)
lr_auc = roc_auc_score(y_test, y_test_pred_proba_lr)

In [ ]:
print(f"Logistic Regression Results:")
print(f"  Precision@{top_k}: {lr_precision:.3f} ({int(lr_precision * top_k)}/{top_k} correct)")
print(f"  Accuracy: {lr_accuracy:.3f}")
print(f"  ROC-AUC: {lr_auc:.3f}")

# <font size="1">
# 
# ### Model 2: Decision Tree
# 
# </font>

In [ ]:
print("MODEL 2: Decision Tree")
print("=" * 60)

dt_model = DecisionTreeClassifier(max_depth=10, min_samples_split=20, random_state=42)
dt_model.fit(X_train, y_train)

y_test_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1]

In [ ]:
test_df_dt = test_df.copy()
test_df_dt['pred_proba'] = y_test_pred_proba_dt
test_sorted_dt = test_df_dt.sort_values('pred_proba', ascending=False)
top_dt = test_sorted_dt.head(top_k)
dt_precision = top_dt['target'].mean()

In [ ]:
dt_accuracy = dt_model.score(X_test, y_test)
dt_auc = roc_auc_score(y_test, y_test_pred_proba_dt)

In [ ]:
print(f"Decision Tree Results:")
print(f"  Precision@{top_k}: {dt_precision:.3f} ({int(dt_precision * top_k)}/{top_k} correct)")
print(f"  Accuracy: {dt_accuracy:.3f}")
print(f"  ROC-AUC: {dt_auc:.3f}")


# <font size="1">
# 
# ### Model 3: Random Forest (Primary Model)
# 
# </font>

In [ ]:
print("MODEL 3: Random Forest (Primary Model)")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

In [ ]:
y_test_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
y_test_pred_rf = rf_model.predict(X_test)


In [ ]:
test_df_rf = test_df.copy()
test_df_rf['pred_proba'] = y_test_pred_proba_rf
test_sorted_rf = test_df_rf.sort_values('pred_proba', ascending=False)
top_rf = test_sorted_rf.head(top_k)
rf_precision = top_rf['target'].mean()


In [ ]:
rf_accuracy = rf_model.score(X_test, y_test)
rf_auc = roc_auc_score(y_test, y_test_pred_proba_rf)


In [ ]:
print(f"Random Forest Results:")
print(f"  Precision@{top_k}: {rf_precision:.3f} ({int(rf_precision * top_k)}/{top_k} correct)")
print(f"  Accuracy: {rf_accuracy:.3f}")
print(f"  ROC-AUC: {rf_auc:.3f}")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': available_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Feature Importances:")
print(feature_importance.head(10).to_string(index=False))

# <font size="1">
# 
# ### Model vs Baseline Comparison
# 
# </font>

In [ ]:
print("MODEL VS BASELINE COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Method': ['Baseline (Hand-coded Rule)', 'Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Precision@50': [
        baseline_precision,
        lr_precision,
        dt_precision,
        rf_precision
    ],
    'Improvement over Baseline': [
        '0.0x',
        f"{lr_precision / baseline_precision:.2f}x",
        f"{dt_precision / baseline_precision:.2f}x",
        f"{rf_precision / baseline_precision:.2f}x"
    ]
})


In [ ]:
print(comparison.to_string(index=False))

In [ ]:
rf_lift = rf_precision / baseline_precision
print(f"\nRandom Forest Lift: {rf_lift:.2f}x better than baseline")
print(f"Random Forest finds {int(rf_precision * top_k) - int(baseline_precision * top_k)} more true positives in the top {top_k}")

In [ ]:
print("\n" + "=" * 60)
print("SUMMARY: Random Forest is the winner")
print("=" * 60)
print(f"Baseline Precision@{top_k}: {baseline_precision:.3f}")
print(f"Random Forest Precision@{top_k}: {rf_precision:.3f}")
print(f"Improvement: {rf_lift:.2f}x")


# <font size="1">
# 
# ## 4. Errors and interpretation
# 
# ### Confusion Matrix
# 
# </font>

In [ ]:
print("CONFUSION MATRIX (Random Forest)")
print("=" * 60)

cm = confusion_matrix(y_test, y_test_pred_rf)
print(cm)

cm_df = pd.DataFrame(cm, columns=['Predicted No', 'Predicted Yes'], index=['Actual No', 'Actual Yes'])
print("\nConfusion Matrix (Actual vs Predicted):")
print(cm_df)

In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

In [ ]:
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

In [ ]:
print(f"\nPrecision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Identify misclassifications
test_df_rf['pred_label'] = y_test_pred_rf
test_df_rf['correct'] = test_df_rf['target'] == test_df_rf['pred_label']
test_df_rf['error_type'] = 'Correct'
test_df_rf.loc[(test_df_rf['target'] == 1) & (test_df_rf['pred_label'] == 0), 'error_type'] = 'False Negative'
test_df_rf.loc[(test_df_rf['target'] == 0) & (test_df_rf['pred_label'] == 1), 'error_type'] = 'False Positive'

print("Error Types:")
print(test_df_rf['error_type'].value_counts())

In [ ]:
# False Negatives
false_negatives = test_df_rf[(test_df_rf['target'] == 1) & (test_df_rf['pred_label'] == 0)]
print(f"\nFalse Negatives (missed declining pages): {len(false_negatives)}")
print("These are pages that are actually declining but the model missed.")


In [ ]:
# False Positives
false_positives = test_df_rf[(test_df_rf['target'] == 0) & (test_df_rf['pred_label'] == 1)]
print(f"\nFalse Positives (wrongly flagged): {len(false_positives)}")
print("These are pages that the model flagged for review but are not actually declining.")

In [ ]:
# Sample false negatives
if len(false_negatives) > 0:
    print("\nSample False Negatives (missed declining pages):")
    cols_to_show = ['content_id', 'impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'pred_proba']
    cols_to_show = [c for c in cols_to_show if c in false_negatives.columns]
    print(false_negatives[cols_to_show].head(5).to_string(index=False))

# Sample false positives
if len(false_positives) > 0:
    print("\nSample False Positives (wrongly flagged):")
    cols_to_show = ['content_id', 'impressions_90d', 'avg_position', 'ctr', 'content_age_days', 'pred_proba']
    cols_to_show = [c for c in cols_to_show if c in false_positives.columns]
    print(false_positives[cols_to_show].head(5).to_string(index=False))


# <font size="1">
# 
# ### What the Model Learned
# 
# </font>

In [ ]:
print("FEATURE IMPORTANCE INTERPRETATION")
print("=" * 60)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

feature_importance['cumulative'] = feature_importance['importance'].cumsum()
print(f"\nTop 5 features account for {feature_importance.head(5)['importance'].sum():.2%} of total importance")
print(f"Top 10 features account for {feature_importance.head(10)['importance'].sum():.2%} of total importance")


# <font size="1">
# 
# ### Model Interpretation: What Do These Features Tell Me?
# 
# **From the feature importance above:**
# 
# 1. **`impressions_90d` is the most important feature** — high-impression pages matter more. A 20% drop on 10,000 impressions is much more impactful than on 100 impressions.
# 
# 2. **`avg_position` matters significantly** — pages losing position are at higher risk. Position tier also matters, confirming non-linearity.
# 
# 3. **`content_age_days` is important** — older pages are more likely to be stale and decline.
# 
# 4. **`ctr` and `engagement_rate` show that user interaction signals matter** — low engagement precedes decline.
# 
# 5. **The model uses a mix of signals** — volume, position, engagement, and age. No single feature dominates completely.
# 
# **What this tells me about the problem:**
# - Decline is multi-factorial. No single signal captures it alone.
# - The model has learned patterns from the data that a hand-coded rule could miss.
# - The feature importance ranking gives me something I can explain to stakeholders.
# 
# **What the errors look like:**
# - **False Negatives (missed declining pages):** These pages were actually declining but the model didn't flag them. Many have moderate volume and position — they fall in the gray area where signals are mixed.
# - **False Positives (wrongly flagged):** These pages were flagged but aren't actually declining. They often have high volume and age but haven't declined yet — they might be "at risk" but not yet declining.
# 
# **How I'd improve:**
# 1. Use a future-looking target instead of current-window proxy
# 2. Add more features: trend slope, seasonal patterns
# 3. Try XGBoost for potential performance gains
# 4. Use the full warehouse dataset for more training data
# 
# </font>

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.